# Inside the inference engine: what makes one replica fast

© 2026, Anyscale. All Rights Reserved

Serving an LLM splits across two layers: a *framework* that coordinates a fleet of replicas, and an *engine* that makes any one replica fast. This notebook is about the engine, and how a single replica turns a GPU into tokens. You will fix the numbers an engine is judged by, meet the key-value cache that decode lives on, see the two phases of a request that load the GPU in opposite ways, and walk through the optimizations that move each number, then measure one of them live on the GPU.

<div class="alert alert-block alert-info">
<b>Roadmap for this notebook</b>
<ol>
    <li>How performance is measured.</li>
    <li>The key-value cache.</li>
    <li>Paged attention.</li>
    <li>Prefix caching.</li>
    <li>Prefill and decode: two phases, two bottlenecks.</li>
    <li>Continuous batching.</li>
    <li>Chunked prefill.</li>
    <li>Measure it on the GPU.</li>
</ol>
</div>

**Imports**

In [ ]:
import time

import requests
import ray
from openai import OpenAI
from ray import serve
from ray.serve.llm import LLMConfig, build_openai_app

## 1. How performance is measured

A classic model server has two performance numbers: how long one request takes, and how many requests per second a replica sustains. Batch size trades one against the other. LLM serving breaks both, because the response arrives one token at a time instead of all at once.

- **Latency is no longer one number**
    - **TTFT (time to first token)** = arrival to the first token: queue wait, then prefill.
    - **ITL / TPOT (inter-token latency / time per output token)** = the gaps between tokens, set by decode.
    - ITL is every gap; TPOT is one request's average, `(E2E - TTFT) / (output tokens - 1)`.
    - **E2E latency** = arrival to the last token: TTFT plus the whole decode.
- **Which latency matters depends on who reads the tokens**
    - A person reading a stream: TTFT, then any gap long enough to notice.
    - A program waiting on a tool call: E2E only, since nothing is usable until the last token.
- **A request is no longer a fixed unit of work**
    - Requests per second still counts completed requests, but only compares runs at the same prompt and output length.
    - Tokens per second counts what the GPU produced, reported as output tokens and as input plus output.
    - A single user's token rate is about `1 / ITL`. More concurrency raises tokens per second and lowers that rate.
- **Two latencies now trade against each other**
    - Prefill and decode hit different hardware limits, so shortening TTFT usually stretches ITL.
    - No engine setting is best for both; you pick the one your target cares about.

The top row is one streamed request: queue wait plus prefill is TTFT, the decode gaps are ITL, and the full bar is E2E. Throughput needs more than one request, so the bottom row shows several running at once.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/nb4_ttft_timeline_rev2.png" loading="lazy" width="840">

Report what ran, on what load, at which percentile, and against which target. Qwen2.5-0.5B on one GPU, **1,000-token prompts and 300-token replies**, targeting p95 TTFT under 500 ms and p95 ITL under 100 ms, reads like this:

| concurrency | p95 TTFT | p95 ITL | output tok/s | req/s | goodput |
|---|---|---|---|---|---|
| 32 | 310 ms | 38 ms | 1,260 | 4.2 | 100% |
| 64 | 890 ms | 61 ms | 1,510 | 5.0 | 41% |

Doubling concurrency buys 20% more tokens per second and misses the TTFT target on more than half the requests. That is what goodput catches and throughput hides.

Measuring one replica, in order:

1. **Set the targets.** For interactive chat: p95 TTFT under 500 ms, p95 ITL between 50 and 100 ms, which is 10 to 20 tokens per second per user.
2. **Take the workload from real traffic.** For example a 1,000-token median prompt and a 300-token median reply. A 128-token synthetic prompt makes every number look better than production.
3. **Sweep concurrency.** Replay that workload at 1, 4, 8, 16 and 32 concurrent, recording p95 TTFT, p95 ITL, requests and tokens per second, and goodput at each.
4. **Find the highest concurrency that still passes.** That is the replica's capacity. Above it, tokens per second keeps rising while requests miss their targets.
5. **Convert capacity to replicas.** Peak demand divided by the requests per second one replica sustained at that concurrency gives the fleet size.
6. **Tune last.** Change engine settings only after you have that baseline, then rerun the same sweep to see whether it moved.

The rest of this notebook is the engine behind those numbers: what it implements for you, and the few knobs it leaves you to trade one number against another. It works in two places, the key-value cache a replica holds and how the scheduler packs work into each engine step, and the sections take them in that order.

## 2. The key-value cache

At every transformer layer, **self-attention** is scaled dot-product attention:

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Generation is autoregressive: the model emits one token at a time, and each new token attends to every token before it. At each step the current query scores every prior key, softmaxs into weights, and blends the value vectors. Two facts about attention make a cache the obvious move.

- **A key and value depend only on their own token.** They are fixed linear projections of that token's hidden state; once computed they never change, however many tokens follow.
- **Every step needs all prior keys and values.** Generating token *t* compares the current query against the keys and values of tokens 1 to *t*.

Without a cache, every step re-projects all prior tokens, so the projection work grows each step (quadratic). With a cache, each token's key and value are computed once and read back from then on (linear).

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/nb4_kv_why_rev1.png" loading="lazy" width="1000">

<div class="alert alert-block alert-warning">
The cache removes the redundant <i>recompute</i>, not the need to <i>look at</i> every prior token: each decode step still reads the whole cache from memory to attend over it. That work is now <b>memory traffic</b> (reading the cache from HBM), not math, which is why decode is bound by memory bandwidth and gets worse at long context.
</div>

The size is one product, and each factor of it maps to one axis of the cache.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/kv_cache_layered_rev1.png" loading="lazy" width="910">

Multiplied out for Qwen2.5-0.5B in bf16, the per-token constant is 12 KB, so one 4096-token sequence carries about 50 MB of key-value cache. The row splits in two:

- **Everything except sequence length and batch is that per-token constant**
    - Fewer key-value heads shrinks it: MHA keeps one per query head, GQA a few (Qwen2.5-0.5B uses 2), MQA a single shared one.
    - So does a smaller key-value dtype, FP8 or INT8.
    - MLA sidesteps the per-head shape altogether: DeepSeek and Kimi cache one compressed latent per token, re-expanded per head at read time.
    - Hybrid stacks cut the layer term instead: on Qwen3.5, linear-attention layers hold fixed-size state, so only one layer in four grows.
- **Sequence length times batch is what grows while you serve**
    - `max_model_len` is the engine setting that caps how many tokens one sequence may hold, so it is the key-value budget you are spending.
    - Raise it and each sequence may claim more, so fewer of them fit at once.

<div class="alert alert-block alert-warning">
If prompt plus generation can exceed <code>max_model_len</code>, vLLM rejects or truncates the request.
</div>

Weights and a headroom reserve are fixed; the key-value cache pool gets whatever is left, and weight precision sets how much that is.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/gpu_mem_partition_rev1.png" loading="lazy" width="1000">

Weights and the key-value cache are two different tenants, and that difference is the whole batching story.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/nb4_two_tenants_rev1.png" loading="lazy" width="1000">

The weights are one fixed block, read once per forward pass; that single read produces a token for every sequence in the batch, so its cost is split across the batch. The key-value cache is the opposite: it is per sequence, so a batch of *B* sequences reads *B* separate caches, each growing with its own context. Batching lifts throughput until those per-sequence cache reads dominate, the point where long context defeats it (section 5).

## 3. Paged attention

The first key-value cache win falls straight out of section 2: stop pre-reserving the whole `max_model_len` for every sequence. Concept and diagram only.

Contiguous reservation leaves a wasted tail per sequence; paged allocation packs scattered fixed blocks behind a block table, wasting at most one partial block each.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/paged_attention_rev1.png" loading="lazy" width="1000">

- **The waste it removes.** A naive layout reserves a contiguous `max_model_len` of key-value per sequence, so every sequence wastes everything between its actual length and the cap. That is internal fragmentation, the wasted tail.
- **The fix.** Split the key-value cache into fixed-size blocks (16 tokens each) allocated on demand; a per-sequence block table maps logical token ranges to scattered physical blocks. A sequence then wastes at most part of one final block.
- **The analogy.** This is OS virtual memory for the key-value cache: pages instead of one big contiguous reservation, so near-zero waste and far more concurrent sequences in the same pool.

## 4. Prefix caching

The second key-value cache win: reuse identical prefix key-value across requests so prefill shrinks to just the new tokens.

Request B shares request A's prefix: only its prefill stub shrinks (the shared-prefix key-value is reused, not recomputed), while the decode is unchanged.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/prefix_cache_rev1.png" loading="lazy" width="960">

- **What it does.** When many requests share a prompt prefix (a system prompt, a few-shot preamble, a long pasted document), its key-value cache is identical every time. Prefix caching keeps that cache and skips recomputing it, so prefill processes only the new tokens.
- **The default.** On by default in vLLM V1; set explicitly below for clarity.
- **The key nuance: it speeds prefill and TTFT, not decode.** Once generation starts, every step still re-reads the weights plus the whole key-value cache. Prefix caching cannot raise decode throughput; it only shrinks the one-shot prefill.
- **Why the cache is per replica.** The win only lands when a request hits a replica that already holds its prefix key-value. Each replica keeps its own cache, so across a multi-replica fleet *which* replica serves a request decides whether the reuse happens at all.

Prefix caching is the vLLM V1 default; this is how you would set it explicitly.

```python
engine_kwargs = {"enable_prefix_caching": True}   # default True in vLLM V1; shown for clarity
```

## 5. Prefill and decode: two phases, two bottlenecks

A GPU is limited by two things: how fast it computes, and how fast it reads bytes from memory. **Arithmetic intensity** is the math done per byte read: high intensity runs into the compute limit, low intensity into the memory limit. Reading the weights costs the same whether the step then computes one token or a thousand:

- **Prefill: one large matrix multiply**
    - The whole prompt goes through each weight matrix at once, building the cache in a single pass.
    - One weight read serves ~1,000 tokens, so ~1,000 FLOPs per byte: compute is the limit, and it sets TTFT.
- **Decode: a matrix times a single vector**
    - One new token per sequence goes through the same weights, extending the cache by one entry.
    - That read serves one token, so ~1 FLOP per byte: memory is the limit, and it sets ITL.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/nb4_roofline_rev1.png" loading="lazy" width="910">

- **Batching helps decode, not prefill.** N sequences share one weight read, so throughput rises at unchanged ITL, while prefill is already compute-limited.
- **Longer context lowers intensity.** Every step re-reads the whole cache at ~1 FLOP per byte, and caches are per sequence, so context limits how far batching helps.
- **Lower precision reads fewer bytes.** Half the bytes per weight halves the read for the same math, so decode benefits most.

## 6. Continuous batching

The first scheduler-level win: keep the batch full so decode never idles. This is the automatic cure for the weight-bandwidth-bound regime, since a fuller batch amortizes the shared weight read across more sequences.

Static batching leaves idle tails as sequences finish early; continuous batching fills each freed slot with a waiting sequence on the next step.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/continuous_batching_rev2.png" loading="lazy" width="900">

- **The problem with static batching.** Launch a batch together and hold every slot until the slowest sequence finishes. Sequences that finish early leave idle slots, so the GPU runs a shrinking batch and wastes bandwidth.
- **Iteration-level batching.** Continuous batching swaps a finished sequence out and admits a waiting one on the very next step, so the batch stays full token by token and throughput stays high.
- **Built in.** vLLM's V1 scheduler does this for you; there is no flag to turn on. It is the default scheduling model.

## 7. Chunked prefill

The second scheduler-level win, and the explicit TTFT-versus-ITL dial: do not let one long prompt's prefill stall everyone's decode.

Each step spends a fixed token budget on the running decodes plus one prefill chunk, spreading a long prompt across consecutive steps instead of one oversized blocking step.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/chunked_prefill_rev1.png" loading="lazy" width="790">

- **The token budget.** Each engine step has a token budget, `max_num_batched_tokens`. Chunked prefill spends most of it on the running decodes and a slice on a long prompt's prefill, so a big prompt is spread across several consecutive steps instead of one giant blocking step.
- **The trade-off dial.** `max_num_batched_tokens` is the throughput-versus-latency knob: a smaller budget protects ITL (decodes keep ticking while prefill drips in), a larger budget favors TTFT and prefill throughput (the prompt clears faster but can stretch the ongoing decodes).

Tune the per-step token budget to choose which number you protect.

```python
engine_kwargs = {"max_num_batched_tokens": 2048}   # smaller -> protect ITL; larger (>8192) -> favor TTFT/throughput
```

<div class="alert alert-block alert-warning">
<code>max_num_batched_tokens</code> is exactly where a single co-located engine is forced to compromise: one budget serves both phases, so there is no setting that is best for both TTFT and ITL. Pick the number your SLO cares about, or split the two phases onto separate pools of GPUs, which is what production fleets do when both targets are hard.
</div>

## 8. Measure it on the GPU

Close the loop empirically and prove the section 4 claim. This is the only live section: serve Qwen2.5-0.5B with prefix caching on, stream a long-shared-prefix request cold then warm, and show TTFT shrinks while decode does not. Then glance at the dashboard those metrics feed.

### 8.1 Serve Qwen2.5-0.5B with prefix caching on

`build_openai_app` turns one `LLMConfig` into an OpenAI-compatible endpoint. Prefix caching is set explicitly here, though it is the vLLM V1 default.

In [ ]:
llm_config = LLMConfig(
    model_loading_config=dict(
        model_id="qwen-0.5b",
        model_source="s3://anyscale-public-materials-use2/models/Qwen/Qwen2.5-0.5B-Instruct",
    ),
    deployment_config=dict(autoscaling_config=dict(min_replicas=1, max_replicas=1)),
    runtime_env=dict(env_vars={"AWS_REGION": "us-east-2"}),   # the region this bucket lives in
    engine_kwargs=dict(
        load_format="runai_streamer",   # required to read the s3:// source
        max_model_len=4096,
        enforce_eager=True,
        enable_prefix_caching=True,
    ),
)
serve.run(build_openai_app({"llm_configs": [llm_config]}), blocking=False)

Note: `load_format="runai_streamer"` is what makes the `s3://` source work. vLLM's Run:ai Model Streamer reads the safetensors straight out of the bucket into GPU memory, skipping a separate download to local disk. The first run streams about 1 GB, so the replica takes a moment to become healthy; once `serve.run` returns, the endpoint is live at `localhost:8000`.

### 8.2 Cold versus warm prefix-cache replay

The shape to expect: only the prefill (time to first token) collapses on the warm call, while the decode stays the same length.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/nb4_cold_vs_warm_rev1.png" loading="lazy" width="920">

Point an OpenAI client at the endpoint and build a long system prefix that both requests will share. The `timed` helper streams one completion and returns the time to first token alongside the total time.

In [ ]:
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
LONG_PREFIX = "You are a careful assistant. Reference table:\n" + \
    "\n".join(f"| {i} | {i * i} |" for i in range(250))   # ~3000-token prefix: long enough to show prefill cost, fits max_model_len=4096 with room for the reply

def timed(user_msg: str, max_tokens: int = 16) -> tuple[float, float]:
    """Stream one completion; return (time-to-first-token, total) in seconds."""
    t0 = time.perf_counter()
    ttft = None
    stream = client.chat.completions.create(
        model="qwen-0.5b", max_tokens=max_tokens, stream=True,
        messages=[{"role": "system", "content": LONG_PREFIX},
                  {"role": "user", "content": user_msg}],
    )
    for chunk in stream:
        if chunk.choices[0].delta.content and ttft is None:
            ttft = time.perf_counter() - t0
    return (ttft or time.perf_counter() - t0), time.perf_counter() - t0

The first call computes the prefix key-value cache (cold); the second reuses it (warm). Both run the same short decode, so the gap is the prefill the cache skipped.

In [ ]:
cold_ttft, cold_total = timed("Summarize the table in one word.")   # populates the prefix cache
warm_ttft, warm_total = timed("Now summarize it in two words.")     # reuses the cached prefix
print(f"cold ttft={cold_ttft*1e3:.0f}ms  warm ttft={warm_ttft*1e3:.0f}ms  speedup={cold_ttft/warm_ttft:.1f}x")

Note: TTFT shrinks; the decode portion does not. This is the section 4 claim, measured.

### 8.3 Glance at the dashboard, then tear down

The engine publishes its metrics into Ray's own metrics system, so they surface on each node's Ray metrics agent rather than on the port that serves the model. Name the two this notebook is about: the key-value pool fullness gauge (section 2) and the prefix-cache query counter that pairs with the hits counter to form a hit rate (section 4).

In [ ]:
def scrape(metric: str) -> list[str]:
    """Prometheus samples for one metric, gathered from every node's Ray metrics agent."""
    text = "".join(
        requests.get(f"http://{n['NodeManagerAddress']}:{n['MetricsExportPort']}/metrics", timeout=10).text
        for n in ray.nodes() if n["Alive"]
    )
    return [ln for ln in text.splitlines() if ln.startswith(metric)]

time.sleep(15)   # Ray's agents republish roughly every 10s; scraping sooner comes back empty

print(scrape("ray_vllm_kv_cache_usage_perc"))         # KV pool fullness gauge (section 2)
print(scrape("ray_vllm_prefix_cache_queries_total"))  # paired with ...prefix_cache_hits_total -> hit rate (section 4)

Note: the dashboard is the picture to remember, not this parse. The hit-rate panel computes `100 * rate(ray_vllm_prefix_cache_hits_total) / rate(ray_vllm_prefix_cache_queries_total)`; a key-value gauge near 100% means you are out of pool room (section 2) and requests start to queue.

<div class="alert alert-block alert-warning">
<b>Scraping the wrong port fails silently.</b> Port 8000 serves only the app's own routes, so <code>/metrics</code> there returns a 404, and <code>requests.get</code> does not raise on a 404. You get an empty list back rather than an error.
</div>

In [ ]:
serve.shutdown()

<div class="alert alert-block alert-info">
<b>Reference implementation:</b> the runnable load generator and the cache-reset helper live in <code>code/llm/engine/load_test.py</code> (TTFT, time per output token, and throughput at bounded concurrency) and <code>code/llm/engine/reset_cache.py</code> (reset the prefix cache, then re-time to make the cache's value visible).
</div>